### Silver — patient standardization and master patient index (MPI).

 Consolidates patient records from three source systems into one golden
 record per real person, with a stable patient_golden_id that survives
 reprocessing.

 This is the highest-risk transformation in the platform. A false merge
 joins two people's medical histories; a missed merge fragments one
 person's care record. Both are clinical safety issues, not merely data
 quality ones — which is why the thresholds are conservative and everything
 between them goes to a human rather than being guessed.

- Input : ehr_patient (~26,000), 
         sched_patient (~20,400),
         fin_patient_account (~16,000)  ->  ~62,400 records
- Output: silver_patient_golden, 
         silver_patient_xref, 
         silver_patient_pii,
         silver_patient_match_review

In [1]:
# PARAMETERS CELL ********************
batch_id = "MPI_MANUAL"
BRONZE = "lh_bronze.dbo"
auto_link_threshold = 14.0
review_threshold = 8.0
hcn_salt = "ohn-dev-salt-change-in-prod"

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 3, Finished, Available, Finished, False)

In [2]:
from datetime import datetime, timezone

from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType

run_ts = datetime.now(timezone.utc)

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 4, Finished, Available, Finished, False)

Source precedence for survivorship tie-breaks. The EHR is the clinical system of record; scheduling and finance capture demographics as a side effect of doing something else, so their values are less trustworthy when two sources disagree and neither is more recent.

In [3]:
SOURCE_RANK = {"EHR": 1, "SCHED": 2, "FIN": 3}
print(f"MPI run {batch_id} at {run_ts.isoformat()}")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 5, Finished, Available, Finished, False)

MPI run MPI_MANUAL at 2026-08-17T11:45:04.112432+00:00


### 1. Project each source onto a common contract
 Each system names its columns differently. Normalising here rather than
 inside the matching logic means adding a fourth source is one more select,
 not a change to the scoring code.

In [4]:
def normalize(df, source_system, m):
    return df.select(
        F.lit(source_system).alias("source_system"),
        F.col(m["id"]).cast(StringType()).alias("source_patient_id"),
        F.col(m["given"]).alias("given_raw"),
        F.col(m["family"]).alias("family_raw"),
        F.col(m["dob"]).alias("dob_raw"),
        F.col(m["sex"]).alias("sex_raw"),
        F.col(m["hcn"]).alias("hcn_raw"),
        F.col(m["postal"]).alias("postal_raw"),
        F.col(m["phone"]).alias("phone_raw"),
        (F.col(m["lang"]) if m.get("lang")
         else F.lit(None).cast(StringType())).alias("language"),
        F.col(m["updated"]).cast("timestamp").alias("source_updated_ts"),
    )
 
 
ehr = normalize(spark.table(f"{BRONZE}.ehr_patient"), "EHR", {
    "id": "patient_id", "given": "first_name", "family": "last_name",
    "dob": "date_of_birth", "sex": "gender", "hcn": "health_card_number",
    "postal": "postal_code", "phone": "primary_phone",
    "lang": "preferred_language", "updated": "last_modified_ts"})
 
sched = normalize(spark.table(f"{BRONZE}.sched_patient"), "SCHED", {
    "id": "patient_ref", "given": "given", "family": "surname",
    "dob": "dob", "sex": "sex", "hcn": "health_card",
    "postal": "postal", "phone": "contact_phone", "updated": "modified_at"})
 
fin = normalize(spark.table(f"{BRONZE}.fin_patient_account"), "FIN", {
    "id": "account_holder_id", "given": "first_nm", "family": "last_nm",
    "dob": "birth_dt", "sex": "gender_cd", "hcn": "hc_number",
    "postal": "zip_postal", "phone": "phone_number", "updated": "update_dt"})
 
records = ehr.unionByName(sched).unionByName(fin)
print(f"Source records: {records.count():,}")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 6, Finished, Available, Finished, False)

Source records: 62,431


## 2. Standardize

 Every rule here exists because the data actually contains that problem.
 Dates arrive in three formats because three systems wrote them. Health
 cards arrive with dashes, with spaces, or with neither.

In [5]:
def std_string(c):
    cleaned = F.upper(F.trim(F.regexp_replace(c, r"\s+", " ")))
    return F.when(cleaned == "", None).otherwise(cleaned)
 
 
def std_postal(c):
    compact = F.upper(F.regexp_replace(c, r"[^A-Za-z0-9]", ""))
    return F.when(compact.rlike(r"^[A-Z]\d[A-Z]\d[A-Z]\d$"), compact).otherwise(None)
 
 
def std_phone(c):
    d = F.regexp_replace(c, r"\D", "")
    return (F.when(F.length(d) == 10, F.concat(F.lit("+1"), d))
             .when((F.length(d) == 11) & (F.substring(d, 1, 1) == "1"),
                   F.concat(F.lit("+"), d))
             .otherwise(None))
 
 
def parse_dob(c):
    # Three formats across three systems. coalesce tries each in turn and
    # yields null if none match, rather than guessing.
    return F.coalesce(
        F.to_date(c, "yyyy-MM-dd"),
        F.to_date(c, "dd/MM/yyyy"),
        F.to_date(c, "MM/dd/yyyy"),
    )
 
 
def valid_hcn(c):
    """Ontario health card: 10 digits passing a mod-10 (Luhn) check."""
    d = F.regexp_replace(c, r"\D", "")
    total = None
    for i in range(10):
        digit = F.substring(d, 10 - i, 1).cast("int")
        contrib = (F.when(F.lit(i % 2 == 1),
                          F.when(digit * 2 > 9, digit * 2 - 9).otherwise(digit * 2))
                    .otherwise(digit))
        total = contrib if total is None else total + contrib
    return (F.length(d) == 10) & (total % 10 == 0)
 
 
std = (
    records
    .withColumn("given_name", std_string(F.col("given_raw")))
    .withColumn("family_name", std_string(F.col("family_raw")))
    .withColumn("birth_date", parse_dob(F.col("dob_raw")))
    .withColumn("postal_code", std_postal(F.col("postal_raw")))
    .withColumn("phone", std_phone(F.col("phone_raw")))
    .withColumn("_hcn_digits", F.regexp_replace(F.col("hcn_raw"), r"\D", ""))
)
 
# A birth date in the future or before 1900 is impossible. Null it rather
# than let it anchor a match — a shared impossible date is not evidence that
# two records describe the same person.
std = std.withColumn(
    "birth_date",
    F.when((F.col("birth_date") > F.current_date())
           | (F.col("birth_date") < F.lit("1900-01-01").cast("date")), None)
     .otherwise(F.col("birth_date")))
 
std = (
    std
    .withColumn("hcn_valid", valid_hcn(F.col("hcn_raw")))
    .withColumn("hcn_clean", F.when(F.col("hcn_valid"), F.col("_hcn_digits")))
    # The token is what reaches Gold. The clear health card number exists
    # only in silver_patient_pii, restricted to stewards.
    .withColumn("hcn_token",
                F.when(F.col("hcn_clean").isNotNull(),
                       F.sha2(F.concat(F.col("hcn_clean"), F.lit(hcn_salt)), 256)))
    .withColumn("fsa", F.substring(F.col("postal_code"), 1, 3))
    .withColumn("family_soundex", F.soundex(F.coalesce(F.col("family_name"), F.lit(""))))
    .withColumn("record_uid", F.concat_ws("|", F.col("source_system"),
                                          F.col("source_patient_id")))
    .withColumn("source_rank",
                F.create_map(*[x for k, v in SOURCE_RANK.items()
                               for x in (F.lit(k), F.lit(v))])[F.col("source_system")])
    .drop("_hcn_digits")
)
 
# --- sex through the reference mapping --------------------------------
sex_map = (
    spark.table(f"{BRONZE}.ref_code_mapping")
    .filter((F.col("domain") == "SEX") & (F.lower(F.col("is_active")) == "true"))
    .select(F.col("source_system").alias("_ms"),
            F.upper(F.trim(F.col("source_code"))).alias("_sc"),
            F.col("standard_code").alias("_std"))
)
 
std = (
    std.join(sex_map,
             (std.source_system == F.col("_ms"))
             & (F.upper(F.trim(F.col("sex_raw"))) == F.col("_sc")), "left")
       .withColumn("sex", F.coalesce(F.col("_std"), F.lit("UNKNOWN")))
       .drop("_ms", "_sc", "_std")
).cache()
 
total = std.count()
print(f"Standardized: {total:,}")
print(f"  valid health card  : {std.filter('hcn_valid').count():,}")
print(f"  no health card     : "
      f"{std.filter(F.col('hcn_raw').isNull() | (F.col('hcn_raw') == '')).count():,}")
print(f"  unparseable DOB    : {std.filter(F.col('birth_date').isNull()).count():,}")
print(f"  unmapped sex code  : {std.filter(F.col('sex') == 'UNKNOWN').count():,}")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 7, Finished, Available, Finished, False)

Standardized: 62,431
  valid health card  : 55,597
  no health card     : 5,722
  unparseable DOB    : 1,122
  unmapped sex code  : 623


## 3. Blocking

 Comparing every record to every other is O(n^2) — 62,000 records is 1.9
billion pairs. Three blocking keys generate candidates instead. A true
 duplicate only has to agree on one of them to be caught, so the union of
 the three gives high recall at a fraction of the cost.

In [6]:
def block(df, key, name):
    return (df.filter(key.isNotNull())
              .withColumn("block_key", F.concat_ws("#", F.lit(name), key))
              .select("record_uid", "block_key"))
 
 
blocks = (
    block(std, F.col("hcn_token"), "HCN")
    .unionByName(block(std, F.concat_ws("#", F.col("family_soundex"),
                                        F.col("birth_date")), "NAME_DOB"))
    .unionByName(block(std, F.concat_ws("#", F.col("fsa"), F.year("birth_date"),
                                        F.substring("given_name", 1, 1)), "GEO_YOB"))
)
 
# Guard against pathological blocks. A block of thousands would explode into
# millions of pairs and almost always indicates a default value masquerading
# as real data, not a genuine cluster of duplicates.
sizes = blocks.groupBy("block_key").count()
usable = sizes.filter(F.col("count").between(2, 200)).select("block_key")
oversized = sizes.filter(F.col("count") > 200)
n_over = oversized.count()
if n_over:
    print(f"WARNING: {n_over} oversized blocks excluded — inspect these")
    oversized.write.format("delta").mode("overwrite") \
             .option("overwriteSchema", "true") \
             .saveAsTable("silver_mdm_oversized_blocks")
 
blocks = blocks.join(usable, "block_key")
 
left = blocks.withColumnRenamed("record_uid", "uid_a")
right = blocks.withColumnRenamed("record_uid", "uid_b")
pairs = (left.join(right, "block_key")
              .filter(F.col("uid_a") < F.col("uid_b"))
              .select("uid_a", "uid_b").distinct())
 
n_pairs = pairs.count()
print(f"Candidate pairs: {n_pairs:,}  "
      f"(vs {total * (total - 1) // 2:,} without blocking)")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 8, Finished, Available, Finished, False)

Candidate pairs: 97,163  (vs 1,948,783,665 without blocking)


## 4. Score

 Fellegi-Sunter style additive weights. Agreement on a strong identifier
 adds weight, disagreement subtracts it, and a missing value contributes
 nothing — an absent phone number is not evidence either way, and treating
 it as disagreement would unfairly penalise sparse records.

In [7]:
a = std.select([F.col(c).alias(f"a_{c}") for c in std.columns])
b = std.select([F.col(c).alias(f"b_{c}") for c in std.columns])
 
scored = (pairs.join(a, F.col("uid_a") == F.col("a_record_uid"))
               .join(b, F.col("uid_b") == F.col("b_record_uid")))
 
 
def name_sim(c1, c2):
    """Normalised edit distance as a Jaro-Winkler stand-in.
 
    Spark has levenshtein built in; true Jaro-Winkler needs a UDF and the
    ranking behaviour is close enough at these thresholds.
    """
    return 1 - (F.levenshtein(c1, c2) /
                F.greatest(F.length(c1), F.length(c2), F.lit(5)))
 
 
scored = (
    scored
    .withColumn("w_hcn",
        F.when(F.col("a_hcn_token").isNull() | F.col("b_hcn_token").isNull(), 0.0)
         .when(F.col("a_hcn_token") == F.col("b_hcn_token"), 12.0).otherwise(-6.0))
    .withColumn("w_dob",
        F.when(F.col("a_birth_date").isNull() | F.col("b_birth_date").isNull(), 0.0)
         .when(F.col("a_birth_date") == F.col("b_birth_date"), 6.0)
         .when((F.year("a_birth_date") == F.year("b_birth_date"))
               & (F.dayofmonth("a_birth_date") == F.month("b_birth_date"))
               & (F.month("a_birth_date") == F.dayofmonth("b_birth_date")), 5.0)
         .when(F.abs(F.datediff("a_birth_date", "b_birth_date")) <= 31, 2.0)
         .otherwise(-5.0))
    .withColumn("w_family",
        F.when(F.col("a_family_name").isNull() | F.col("b_family_name").isNull(), 0.0)
         .when(name_sim(F.col("a_family_name"), F.col("b_family_name")) >= 0.90, 4.0)
         .otherwise(-3.0))
    .withColumn("w_given",
        F.when(F.col("a_given_name").isNull() | F.col("b_given_name").isNull(), 0.0)
         .when(name_sim(F.col("a_given_name"), F.col("b_given_name")) >= 0.88, 3.0)
         .otherwise(-2.0))
    .withColumn("w_sex",
        F.when((F.col("a_sex") == "UNKNOWN") | (F.col("b_sex") == "UNKNOWN"), 0.0)
         .when(F.col("a_sex") == F.col("b_sex"), 1.0).otherwise(-3.0))
    .withColumn("w_postal",
        F.when(F.col("a_postal_code").isNull() | F.col("b_postal_code").isNull(), 0.0)
         .when(F.col("a_postal_code") == F.col("b_postal_code"), 3.0)
         .when(F.col("a_fsa") == F.col("b_fsa"), 1.0).otherwise(-1.0))
    .withColumn("w_phone",
        F.when(F.col("a_phone").isNull() | F.col("b_phone").isNull(), 0.0)
         .when(F.col("a_phone") == F.col("b_phone"), 2.5).otherwise(-0.5))
)
 
scored = scored.withColumn(
    "match_score",
    F.col("w_hcn") + F.col("w_dob") + F.col("w_family") + F.col("w_given")
    + F.col("w_sex") + F.col("w_postal") + F.col("w_phone")).cache()

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 9, Finished, Available, Finished, False)

## 5. Apply steward overrides, then decide

 A steward's decision always beats the algorithm and must survive every
 future run — otherwise stewards redo the same work nightly.

In [8]:
if spark.catalog.tableExists("silver_patient_match_override"):
    ov = spark.table("silver_patient_match_override").select(
        F.col("uid_a").alias("o_a"), F.col("uid_b").alias("o_b"), "decision")
    scored = scored.join(ov, (F.col("uid_a") == F.col("o_a"))
                             & (F.col("uid_b") == F.col("o_b")), "left")
else:
    scored = scored.withColumn("decision", F.lit(None).cast(StringType()))
 
scored = scored.withColumn(
    "final_decision",
    F.when(F.col("decision") == "MATCH", "AUTO_LINK")
     .when(F.col("decision") == "NO_MATCH", "DISTINCT")
     .when(F.col("match_score") >= auto_link_threshold, "AUTO_LINK")
     .when(F.col("match_score") >= review_threshold, "REVIEW")
     .otherwise("DISTINCT")).cache()
 
decisions = {r["final_decision"]: r["count"]
             for r in scored.groupBy("final_decision").count().collect()}
print("Pair decisions:", decisions)
 
# The review queue is a deliverable, not a dead end. These are pairs where
# the evidence genuinely does not settle it and a human has to look.
(scored.filter(F.col("final_decision") == "REVIEW")
       .select("uid_a", "uid_b", "match_score",
               "a_given_name", "a_family_name", "a_birth_date", "a_postal_code",
               "b_given_name", "b_family_name", "b_birth_date", "b_postal_code",
               F.lit(batch_id).alias("batch_id"),
               F.lit(run_ts).alias("queued_ts"),
               F.lit("PENDING").alias("review_status"))
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable("silver_patient_match_review"))

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 10, Finished, Available, Finished, False)

Pair decisions: {'REVIEW': 4328, 'DISTINCT': 47334, 'AUTO_LINK': 45528}


## 6. Cluster into identities

 Auto-linked pairs form graph edges; each connected component is one person.
 This handles transitivity: if A matches B and B matches C, all three are
 the same person even when A and C were never directly compared.

 Implemented as iterative label propagation rather than GraphFrames, which
 is not reliably available on the Fabric runtime. Each pass pushes the
 minimum label across every edge; the components are stable once no label
 changes.

In [9]:
edges = (scored.filter(F.col("final_decision") == "AUTO_LINK")
               .select(F.col("uid_a").alias("src"), F.col("uid_b").alias("dst")))
# Undirected: add the reverse of every edge so labels flow both ways.
edges = (edges.unionByName(edges.select(F.col("dst").alias("src"),
                                        F.col("src").alias("dst")))
              .distinct().cache())
print(f"Edges: {edges.count():,}")
 
labels = std.select(F.col("record_uid").alias("id"),
                    F.col("record_uid").alias("component"))
 
for i in range(20):
    propagated = (labels.alias("l")
        .join(edges.alias("e"), F.col("l.id") == F.col("e.src"))
        .select(F.col("e.dst").alias("id"), F.col("l.component").alias("component")))
    updated = (labels.unionByName(propagated)
                     .groupBy("id").agg(F.min("component").alias("component")))
    updated = updated.cache()
    changed = (updated.alias("u").join(labels.alias("p"), "id")
               .filter(F.col("u.component") != F.col("p.component")).count())
    labels = updated
    print(f"  iteration {i + 1}: {changed:,} labels changed")
    if changed == 0:
        break
 
n_components = labels.select("component").distinct().count()
print(f"Distinct identities: {n_components:,} (from {total:,} source records)")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 11, Finished, Available, Finished, False)

Edges: 91,056
  iteration 1: 34,068 labels changed
  iteration 2: 926 labels changed
  iteration 3: 2 labels changed
  iteration 4: 0 labels changed
Distinct identities: 27,905 (from 62,431 source records)


## 7. Assign stable golden IDs

 A new cluster gets a deterministic id derived from its component, so a
 rerun produces the same ids rather than a fresh set. A cluster overlapping
an existing one keeps the existing id, so downstream surrogate keys never
 churn. When two previously separate clusters merge, the retired id is
 recorded so facts pointing at it can be repointed rather than orphaned.

In [10]:
if spark.catalog.tableExists("silver_patient_xref"):
    existing = spark.table("silver_patient_xref").select("record_uid", "patient_golden_id")
else:
    existing = spark.createDataFrame(
        [], "record_uid string, patient_golden_id string")
 
clustered = (labels.withColumnRenamed("id", "record_uid")
                   .join(existing, "record_uid", "left"))
 
w = Window.partitionBy("component").orderBy(F.col("patient_golden_id").asc_nulls_last())
resolved = clustered.withColumn(
    "surviving", F.first("patient_golden_id", ignorenulls=True).over(w))
 
resolved = resolved.withColumn(
    "patient_golden_id_final",
    F.coalesce(F.col("surviving"),
               F.sha2(F.concat(F.lit("OHN"), F.col("component")), 256).substr(1, 36)))
 
retired = (resolved
    .filter(F.col("patient_golden_id").isNotNull()
            & (F.col("patient_golden_id") != F.col("patient_golden_id_final")))
    .select(F.col("patient_golden_id").alias("retired_golden_id"),
            F.col("patient_golden_id_final").alias("surviving_golden_id"),
            F.lit(batch_id).alias("batch_id"), F.lit(run_ts).alias("retired_ts"))
    .distinct())
 
n_retired = retired.count()
if n_retired:
    print(f"Golden IDs retired by cluster merge: {n_retired:,}")
    retired.write.format("delta").mode("append").option("mergeSchema", "true") \
           .saveAsTable("silver_patient_xref_history")
 
xref = resolved.select("record_uid",
                       F.col("patient_golden_id_final").alias("patient_golden_id"),
                       F.lit(batch_id).alias("batch_id"),
                       F.lit(run_ts).alias("updated_ts"))
xref.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_patient_xref")
xref = spark.table("silver_patient_xref")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 12, Finished, Available, Finished, False)

Golden IDs retired by cluster merge: 936


## 8. Survivorship

Attribute-level, not record-level. The freshest address and the most
 reliable health card number often come from different systems, so picking
 one "best" source record wholesale would discard good data. For each field
 the most recently updated non-null value wins, with source precedence
 breaking ties.

In [11]:
enriched = std.join(xref.select("record_uid", "patient_golden_id"), "record_uid")
 
 
def survive(col):
    w_attr = (Window.partitionBy("patient_golden_id")
              .orderBy(F.when(F.col(col).isNull(), 1).otherwise(0),
                       F.col("source_updated_ts").desc_nulls_last(),
                       F.col("source_rank").asc()))
    return F.first(F.col(col), ignorenulls=True).over(w_attr).alias(col)
 
 
golden = (enriched.select(
        "patient_golden_id",
        survive("given_name"), survive("family_name"), survive("birth_date"),
        survive("sex"), survive("hcn_token"), survive("hcn_clean"),
        survive("postal_code"), survive("fsa"), survive("phone"), survive("language"),
        F.first("source_patient_id").over(
            Window.partitionBy("patient_golden_id").orderBy(F.col("source_rank").asc())
        ).alias("source_patient_id"))
    .dropDuplicates(["patient_golden_id"]))
 
# Cluster confidence is the weakest link, not the average. A cluster held
# together by one marginal pair IS marginal, and a mean would hide exactly
# the clusters a steward should be looking at.
conf = (scored.filter(F.col("final_decision") == "AUTO_LINK")
        .select(F.col("uid_a").alias("record_uid"), "match_score")
        .join(xref.select("record_uid", "patient_golden_id"), "record_uid")
        .groupBy("patient_golden_id").agg(F.min("match_score").alias("match_confidence")))
 
counts = (enriched.groupBy("patient_golden_id")
          .agg(F.count("*").alias("source_record_count"),
               F.concat_ws(",", F.sort_array(F.collect_set("source_system")))
                .alias("contributing_sources")))
 
age_yrs = F.floor(F.months_between(F.current_date(), F.col("birth_date")) / 12)
 
golden = (golden.join(conf, "patient_golden_id", "left")
                .join(counts, "patient_golden_id", "left")
                # 99.0 marks a singleton: no pair had to be judged, so
                # confidence is not merely high, it is not applicable.
                .withColumn("match_confidence",
                            F.coalesce(F.col("match_confidence"), F.lit(99.0)))
                .withColumn("birth_year", F.year("birth_date"))
                .withColumn("age_band",
                    F.when(F.col("birth_date").isNull(), "Unknown")
                     .when(age_yrs < 18, "0-17").when(age_yrs < 35, "18-34")
                     .when(age_yrs < 50, "35-49").when(age_yrs < 65, "50-64")
                     .when(age_yrs < 75, "65-74").when(age_yrs < 85, "75-84")
                     .otherwise("85+"))
                .withColumn("is_deceased", F.lit(False))
                .withColumn("_batch_id", F.lit(batch_id))
                .withColumn("_updated_ts", F.lit(run_ts)))
 
# Direct identifiers split out here. Gold never sees a clear name or health
# card number — only the token and the age band.
(golden.select("patient_golden_id", "given_name", "family_name", "birth_date",
               "hcn_clean", "postal_code", "phone", "_updated_ts")
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable("silver_patient_pii"))
 
(golden.drop("given_name", "family_name", "hcn_clean", "phone", "postal_code")
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable("silver_patient_golden"))
 
n_golden = spark.table("silver_patient_golden").count()
print(f"\nGolden patients: {n_golden:,}  (from {total:,} source records, "
      f"ratio {total / max(1, n_golden):.2f})")

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 13, Finished, Available, Finished, False)


Golden patients: 27,905  (from 62,431 source records, ratio 2.24)


## 9. SUMMARY

In [12]:
g = spark.table("silver_patient_golden")
 
print("Records merged per golden patient:")
g.groupBy("source_record_count").count().orderBy("source_record_count").show()
 
print("Contributing source combinations:")
g.groupBy("contributing_sources").count().orderBy(F.desc("count")).show(truncate=False)
 
print("Match confidence:")
(g.withColumn("band",
    F.when(F.col("match_confidence") >= 99, "singleton (no pair judged)")
     .when(F.col("match_confidence") >= 20, "very high")
     .when(F.col("match_confidence") >= auto_link_threshold, "at auto-link threshold")
     .otherwise("below threshold — investigate"))
 .groupBy("band").count().orderBy(F.desc("count")).show(truncate=False))

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 14, Finished, Available, Finished, False)

Records merged per golden patient:
+-------------------+-----+
|source_record_count|count|
+-------------------+-----+
|                  1| 5730|
|                  2|10283|
|                  3|11439|
|                  4|  448|
|                  5|    4|
|                  6|    1|
+-------------------+-----+

Contributing source combinations:
+--------------------+-----+
|contributing_sources|count|
+--------------------+-----+
|EHR,FIN,SCHED       |11457|
|EHR,SCHED           |7256 |
|EHR                 |3260 |
|EHR,FIN             |3068 |
|SCHED               |1405 |
|FIN                 |1134 |
|FIN,SCHED           |325  |
+--------------------+-----+

Match confidence:
+--------------------------+-----+
|band                      |count|
+--------------------------+-----+
|very high                 |14751|
|at auto-link threshold    |7424 |
|singleton (no pair judged)|5730 |
+--------------------------+-----+



In [13]:
import json
mssparkutils.notebook.exit(json.dumps({
    "batch_id": batch_id,
    "source_records": total,
    "golden_patients": n_golden,
    "pairs_compared": n_pairs,
    "auto_linked": decisions.get("AUTO_LINK", 0),
    "review_queue": decisions.get("REVIEW", 0),
    "retired_ids": n_retired,
}))

StatementMeta(, ee805572-14fb-4158-8808-93f8a843f807, 15, Finished, Available, Finished, False)

ExitValue: {"batch_id": "MPI_MANUAL", "source_records": 62431, "golden_patients": 27905, "pairs_compared": 97163, "auto_linked": 45528, "review_queue": 4328, "retired_ids": 936}

In [12]:
from pyspark.sql import functions as F, Window
BRONZE = "lh_bronze.dbo"
SILVER = "lh_silver.dbo"
GOLD = "lh_gold.dbo"

StatementMeta(, 7b791584-8635-4ef8-8aac-f37846737d98, 30, Finished, Available, Finished, False)

In [13]:
tables_df = spark.sql(f"SHOW TABLES IN {GOLD}")

StatementMeta(, 7b791584-8635-4ef8-8aac-f37846737d98, 31, Finished, Available, Finished, False)

In [14]:
# In Databricks, Fabric, or Azure Synapse Notebooks
display(tables_df)

# Or convert to Pandas (displays full text cleanly in Jupyter/notebooks)
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

tables_df.toPandas()

StatementMeta(, 7b791584-8635-4ef8-8aac-f37846737d98, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0203eec8-aaa5-4f3e-b89a-13664915df85)

,namespace,tableName,isTemporary
0,`OHN-dev`.lh_gold.dbo,bridge_encounter_diagnosis,False
1,`OHN-dev`.lh_gold.dbo,dim_admission_type,False
2,`OHN-dev`.lh_gold.dbo,dim_appointment_status,False
3,`OHN-dev`.lh_gold.dbo,dim_bed,False
4,`OHN-dev`.lh_gold.dbo,dim_claim_status,False
5,`OHN-dev`.lh_gold.dbo,dim_date,False
6,`OHN-dev`.lh_gold.dbo,dim_department,False
7,`OHN-dev`.lh_gold.dbo,dim_diagnosis,False
8,`OHN-dev`.lh_gold.dbo,dim_discharge_disposition,False
9,`OHN-dev`.lh_gold.dbo,dim_doctor,False
